In [1]:
import anndata as ad
import numpy as np
import os
import pandas as pd
import scanpy as sc

In [2]:
seed_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/seed/baf_afc/afc.counts.cell_anno.h5ad'
rs_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim/baf_afc/afc.counts.cell_anno.h5ad'
scrs_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/baf_afc/afc.counts.cell_anno.h5ad'

out_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/rs_benchmark/baf/gene/base/pp'

In [3]:
normal_cell_type = "normal"
tumor_cell_type = "tumor"

# Load Data

In [4]:
os.makedirs(out_dir, exist_ok = True)

## Raw seed data

In [5]:
seed = ad.read_h5ad(seed_fn)
seed

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

In [6]:
seed = seed[:, ~seed.var["feature"].duplicated(keep = False)].copy()
seed = seed[seed.obs["cell_type"] == normal_cell_type, :].copy()
seed.obs.index = seed.obs["cell"]
seed.var.index = seed.var["feature"]
seed

AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

## `rs` and `scReadSim`

In [7]:
rs = ad.read_h5ad(rs_fn)
rs

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

In [8]:
rs.obs.index = rs.obs["cell"]
rs.var.index = rs.var["feature"]
rs = rs[:, ~rs.var["feature"].duplicated(keep = False)].copy()
rs

AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

In [9]:
scrs = ad.read_h5ad(scrs_fn)
scrs

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

In [10]:
scrs.obs.index = scrs.obs["cell"]
scrs.var.index = scrs.var["feature"]
scrs = scrs[:, ~scrs.var["feature"].duplicated(keep = False)].copy()
scrs

AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

## Use intersect genes

In [11]:
adata_list = [seed, rs, scrs]

genes = None
for i, adata in enumerate(adata_list):
    if i == 0:
        genes = adata.var["feature"].to_numpy()
    else:
        genes = np.intersect1d(genes, adata.var["feature"])
genes.shape

(32295,)

### Use the same order of genes

In [12]:
seed = seed[:, seed.var["feature"].isin(genes)].copy()
print(seed)

rs = rs[:, seed.var.index].copy()
print(rs)

scrs = scrs[:, seed.var.index].copy()
print(scrs)

AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'


# Split by cell type

In [13]:
seed_normal = seed[seed.obs["cell_type"] == normal_cell_type, :].copy()
print(seed_normal)

rs_normal = rs[rs.obs["cell_type"] == normal_cell_type, :].copy()
print(rs_normal)

rs_tumor = rs[rs.obs["cell_type"] == tumor_cell_type, :].copy()
print(rs_tumor)

scrs_normal = scrs[scrs.obs["cell_type"] == normal_cell_type, :].copy()
print(scrs_normal)

scrs_tumor = scrs[scrs.obs["cell_type"] == tumor_cell_type, :].copy()
print(scrs_tumor)

AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'


# Save Data

## Check the order of cells and genes

In [14]:
assert np.all(rs_normal.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())
assert np.all(rs_tumor.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())
assert np.all(scrs_normal.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())
assert np.all(scrs_tumor.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())

In [15]:
seed_normal.write_h5ad(os.path.join(out_dir, "seed_normal.h5ad"), compression = 'gzip')
rs_normal.write_h5ad(os.path.join(out_dir, "rs_normal.h5ad"), compression = 'gzip')
rs_tumor.write_h5ad(os.path.join(out_dir, "rs_tumor.h5ad"), compression = 'gzip')
scrs_normal.write_h5ad(os.path.join(out_dir, "scrs_normal.h5ad"), compression = 'gzip')
scrs_tumor.write_h5ad(os.path.join(out_dir, "scrs_tumor.h5ad"), compression = 'gzip')